# 🇻🇳 ViMind 2.0: Quy Trình Tinh Chỉnh SFT & Căn Chỉnh DPO (Kiến Trúc 64M)
Notebook này được thiết kế và tối ưu hóa chuyên biệt cho Kaggle GPU (Tesla T4 16GB VRAM):
1. **Nạp trọng số ViMind 2.0 Base (64M)** đã tiền huấn luyện từ Kaggle Dataset `cnghunhlquc/vimind-pretrained-base`.
2. **Supervised Fine-Tuning (SFT):** Tinh chỉnh 2 Epochs với FP16 Mixed Precision trên tập hội thoại tiếng Việt (`dataset/sft_vi.jsonl`).
3. **Thử nghiệm đối thoại nhanh:** Đánh giá năng lực sinh văn bản tiếng Việt sau SFT.
4. **Direct Preference Optimization (DPO):** Căn chỉnh sở thích con người & giảm thiểu ảo giác (`dataset/dpo_vi.jsonl`).
5. **Xuất xưởng thành phẩm:** Đóng gói toàn bộ mô hình (`model.safetensors`, `config.json`, `tokenizer.json`, ...) ra `/kaggle/working/vimind_64m_final` sẵn sàng tải về hoặc triển khai.

In [ ]:
# 1. Clone toàn bộ mã nguồn ViMind từ GitHub và di chuyển vào thư mục làm việc
!rm -rf /kaggle/working/vimind
!git clone https://github.com/WuKong0601/ViMind.git /kaggle/working/vimind
%cd /kaggle/working/vimind


In [ ]:
# 2. Cài đặt các thư viện cần thiết
!pip install -r requirements.txt


In [ ]:
# 3. Kiểm tra thông số phần cứng GPU & Khả năng tương thích CUDA
!nvidia-smi
import torch
print(f'PyTorch Version: {torch.__version__}')
print(f'CUDA Available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    cap = torch.cuda.get_device_capability(0)
    print(f'GPU Device: {gpu_name} (CUDA Capability {cap[0]}.{cap[1]})')


In [ ]:
# 4. Tải dữ liệu SFT & DPO (và Pre-train dataset nếu cần fallback)
!python data_pipeline/download_sft.py
!python data_pipeline/download_dpo.py

import os
has_base = any(
    os.path.exists(p) and (os.path.exists(os.path.join(p, 'model.safetensors')) or os.path.exists(os.path.join(p, 'config.json')))
    for p in ['/kaggle/input/vimind-pretrained-base', '/kaggle/input/vimind-pretrained-base/vimind_64m_final', 'out/vimind_64m_final']
)
if not has_base:
    print('📥 Chưa có base checkpoint, tải dữ liệu Pretraining...')
    !python data_pipeline/download_textbooks_and_books.py --output_path dataset/pretrain_vi_v2.jsonl --max_textbook 100000 --max_books 10000 --max_wiki 100000
else:
    print('⚡ Đã có checkpoint ViMind 2.0 (64M) Base, bỏ qua tải pretrain dataset!')

In [ ]:
# 5. Chạy bộ kiểm thử toàn diện & Kiểm tra kiến trúc ViMind 2.0 (64M)
!python trainer/test_64m_dry_run.py
!python trainer/test_pipeline.py


In [ ]:
# 6. [GIAI ĐOẠN PRE-TRAINING VIMIND 2.0 (64M)]
import os, shutil

base_model_path = None
candidates_64m = [
    '/kaggle/input/vimind-pretrained-base',
    '/kaggle/input/vimind-pretrained-base/vimind_64m_final',
    '/kaggle/input/vimind-64m-checkpoint/vimind_64m_final',
    'out/vimind_64m_final'
]
for p in candidates_64m:
    if os.path.exists(p) and (os.path.exists(os.path.join(p, 'model.safetensors')) or os.path.exists(os.path.join(p, 'config.json'))):
        base_model_path = p
        break

if base_model_path:
    print(f'✅ Đã tìm thấy trọng số ViMind 2.0 (64M) tại: {base_model_path}')
    if base_model_path != 'out/vimind_64m_final':
        os.makedirs('out/vimind_64m_final', exist_ok=True)
        for fname in os.listdir(base_model_path):
            src = os.path.join(base_model_path, fname)
            dst = os.path.join('out/vimind_64m_final', fname)
            if os.path.isfile(src):
                shutil.copy2(src, dst)
        base_model_path = 'out/vimind_64m_final'
    print('⚡ Bỏ qua Pre-training (đã có checkpoint 64M) và chuyển tiếp sang SFT!')
else:
    print('🚀 Không tìm thấy Base Checkpoint, bắt đầu huấn luyện tiền kỳ ViMind 2.0 (64M)...')
    data_file = 'dataset/pretrain_vi_v2.jsonl' if os.path.exists('dataset/pretrain_vi_v2.jsonl') else 'dataset/pretrain_vi.jsonl'
    !python -u trainer/pretrain.py \
        --model_size 64m \
        --data_path {data_file} \
        --tokenizer_dir model \
        --save_dir out \
        --save_weight vimind_64m \
        --epochs 3 \
        --batch_size 16 \
        --accumulation_steps 8 \
        --gradient_checkpointing \
        --learning_rate 5e-4 \
        --dtype float16 \
        --log_interval 50 \
        --save_interval 1000
    base_model_path = 'out/vimind_64m_final'

In [ ]:
# 7. [GIAI ĐOẠN SFT] Tinh chỉnh chỉ dẫn từ trọng số Pre-training để biến thành Chatbot
import os, shutil

sft_model_path = 'out/sft/vimind_64m_sft_final'
if os.path.exists(sft_model_path) and (os.path.exists(os.path.join(sft_model_path, 'model.safetensors')) or os.path.exists(os.path.join(sft_model_path, 'config.json'))):
    print(f'✅ Đã tìm thấy SFT Checkpoint 64M tại: {sft_model_path}')
    print('⚡ Bỏ qua giai đoạn SFT và chuyển ngay sang DPO Alignment!')
else:
    print(f'🚀 Bắt đầu SFT Fine-Tuning ViMind 2.0 (64M) với trọng số nền: {base_model_path}...')
    !python -u trainer/train_sft.py \
        --data_path dataset/sft_vi.jsonl \
        --tokenizer_dir model \
        --from_pretrained {base_model_path} \
        --save_dir out/sft \
        --save_weight vimind_64m_sft \
        --batch_size 16 \
        --accumulation_steps 4 \
        --epochs 2 \
        --learning_rate 1e-4 \
        --dtype float16 \
        --log_interval 25 \
        --save_interval 500

In [ ]:
# 8. [KIỂM THỬ THÀNH PHẨM SFT] Trò chuyện thử với mô hình ViMind 2.0 (64M) SFT
import os
import torch
from transformers import AutoTokenizer
from model.model import ViMindForCausalLM

sft_eval_path = 'out/sft/vimind_64m_sft_final' if os.path.exists('out/sft/vimind_64m_sft_final') else ('out/sft/vimind_64m_sft_step_500' if os.path.exists('out/sft/vimind_64m_sft_step_500') else base_model_path)
if os.path.exists(sft_eval_path):
    tokenizer = AutoTokenizer.from_pretrained('model')
    model = ViMindForCausalLM.from_pretrained(sft_eval_path).cuda()
    model.eval()

    test_prompts = [
        'Xin chào, bạn là ai và bạn có thể giúp gì cho tôi?',
        'Thủ đô của Việt Nam là gì?',
        'Hãy nêu 3 lợi ích của việc tập thể dục mỗi ngày.',
        'Làm thế nào để học lập trình Python hiệu quả?'
    ]

    print('=' * 60)
    print(f'🎉 KẾT QUẢ TRẢ LỜI CỦA VIMIND 2.0 SFT ({sft_eval_path}):')
    print('=' * 60)
    for p in test_prompts:
        formatted = tokenizer.apply_chat_template([{'role': 'user', 'content': p}], tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(formatted, return_tensors='pt').input_ids.cuda()
        with torch.no_grad():
            outputs = model.generate(
                inputs,
                max_new_tokens=150,
                temperature=0.7,
                top_p=0.85,
                top_k=20,
                eos_token_id=tokenizer.eos_token_id,
                pad_token_id=tokenizer.pad_token_id
            )
        ans = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
        print(f'\n👤 Người dùng: {p}')
        print(f'🤖 ViMind 2.0 SFT: {ans.strip()}')
        print('-' * 60)
else:
    print(f'⚠️ Checkpoint not found at: {sft_eval_path}')

In [ ]:
# 9. [GIAI ĐOẠN DPO] Căn chỉnh sở thích con người & Giảm thiểu ảo giác (ViMind 2.0 DPO)
import os
sft_model_path = 'out/sft/vimind_64m_sft_final'
if not os.path.exists(sft_model_path):
    sft_model_path = base_model_path

print(f'🚀 Bắt đầu huấn luyện DPO Alignment với trọng số SFT 64M: {sft_model_path}...')
!python -u trainer/train_dpo.py \
    --data_path dataset/dpo_vi.jsonl \
    --model_path {sft_model_path} \
    --save_dir out/dpo \
    --save_weight vimind_64m_dpo \
    --batch_size 8 \
    --accumulation_steps 4 \
    --epochs 1 \
    --learning_rate 2e-5 \
    --beta 0.1 \
    --max_seq_len 512 \
    --fp16 \
    --log_interval 25 \
    --save_interval 500

In [ ]:
# 10. [KIỂM THỬ THÀNH PHẨM VIMIND 2.0 DPO & XUẤT XƯỞNG]
import os
import shutil
import torch
from transformers import AutoTokenizer
from model.model import ViMindForCausalLM

final_model_path = 'out/dpo/vimind_64m_dpo_final' if os.path.exists('out/dpo/vimind_64m_dpo_final') else 'out/sft/vimind_64m_sft_final'
if os.path.exists(final_model_path):
    tokenizer = AutoTokenizer.from_pretrained('model')
    model = ViMindForCausalLM.from_pretrained(final_model_path).cuda()
    model.eval()

    test_prompts = [
        'Xin chào, bạn là ai và bạn có thể giúp gì cho tôi?',
        'Thủ đô của Việt Nam là gì?',
        'Hãy giải thích ngắn gọn trí tuệ nhân tạo là gì?',
        'Làm thế nào để học lập trình Python hiệu quả?'
    ]

    print('=' * 60)
    print(f'🎉 KẾT QUẢ TRẢ LỜI CỦA VIMIND 2.0 THÀNH PHẨM ({final_model_path}):')
    print('=' * 60)
    for p in test_prompts:
        formatted = tokenizer.apply_chat_template([{'role': 'user', 'content': p}], tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(formatted, return_tensors='pt').input_ids.cuda()
        with torch.no_grad():
            outputs = model.generate(
                inputs,
                max_new_tokens=150,
                temperature=0.6,
                top_p=0.85,
                top_k=20,
                eos_token_id=tokenizer.eos_token_id,
                pad_token_id=tokenizer.pad_token_id
            )
        ans = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
        print(f'\n👤 Người dùng: {p}')
        print(f'🤖 ViMind 2.0: {ans.strip()}')
        print('-' * 60)

    # Xuất xưởng toàn bộ thư mục sang /kaggle/working/vimind_64m_final để tải về trực tiếp
    export_dir = '/kaggle/working/vimind_64m_final'
    os.makedirs(export_dir, exist_ok=True)
    for fname in os.listdir(final_model_path):
        src = os.path.join(final_model_path, fname)
        dst = os.path.join(export_dir, fname)
        if os.path.isfile(src):
            shutil.copy2(src, dst)
    print(f'\n📦 Đã xuất xưởng toàn bộ mô hình thành phẩm ra: {export_dir}')
    print(f'📂 Danh sách tệp tin thành phẩm: {os.listdir(export_dir)}')
else:
    print(f'⚠️ Checkpoint not found at: {final_model_path}')